## Pydantic 資料驗證與多來源欄位標準化對齊

- 目標：精通 **Pydantic V2 的進階型別檢查與欄位自訂驗證邏輯**。學會編寫 standardize_columns() 函數，將來自不同機台、不同測試廠（OSAT）格式不一的「髒欄位」統一對齊標準化，並透過 Pydantic 在最前端擋掉非法的製程數據。


### 1. Pydantic V2 型別檢查與複雜欄位驗證

- 實作：在測試資料管道中，常會收到各種格式。我們需要設計一個全能的資料驗證模型，除了基本**型別轉換，還要阻擋不合規的異常數據**（例如：負的量測數值，或是未依格式命名的 Lot ID）。


In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from datetime import datetime
from typing import Optional, List


class DeviceTestMeasurement(BaseModel):
    """晶粒電性測試點位資料模型"""

    lot_id: str
    wafer_no: int = Field(..., ge=1, le=25)  # 晶圓號必須在 1 到 25 之間
    die_x: int = Field(..., description="晶粒 X 座標")
    die_y: int = Field(..., description="晶粒 Y 座標")
    frequency_ghz: float = Field(..., gt=0.0)  # 測試頻率必須大於 0
    leakage_current_ma: float = Field(..., ge=0.0)  # 漏電流不能為負值
    test_timestamp: datetime = Field(default_factory=datetime.utcnow)

    # 1欄位驗證器 (Field Validator)：檢查 Lot ID 格式
    @field_validator("lot_id")
    @classmethod
    def verify_lot_id_format(cls, v: str) -> str:
        if not v.upper().startswith("LOT_"):
            raise ValueError("Lot ID 必須符合正規半導體命名規範，以 'LOT_' 開頭")
        return v.upper()

    # 2模型驗證器 (Model Validator)：多欄位交叉檢查
    @model_validator(mode="after")
    def check_leakage_and_frequency(self) -> "DeviceTestMeasurement":
        # 實戰商業邏輯：高頻晶片如果漏電流過高，通常頻率表現會受限，藉此找出矛盾數據
        if self.leakage_current_ma > 1.0 and self.frequency_ghz > 35.0:
            raise ValueError(
                f"數據異常矛盾：漏電流過高 ({self.leakage_current_ma} mA) 時，不可能達到高頻 {self.frequency_ghz} GHz"
            )
        return self


# ---- 測試資料驗證機制 ----
print(">>> 測試情境 A：合法資料傳入（包含自動字串轉型）")
valid_data = DeviceTestMeasurement(
    lot_id="lot_npi_01",  # 故意用小寫
    wafer_no="05",  # 故意傳字串數字
    die_x=12,
    die_y=34,
    frequency_ghz=28.5,
    leakage_current_ma=0.05,
)
print(
    f"驗證通過。標準化後的 Lot ID: {valid_data.lot_id}, Wafer No 型別: {type(valid_data.wafer_no)}\n"
)

print(">>> 測試情境 B：非法資料攔截（觸發驗證錯誤）")
try:
    invalid_data = DeviceTestMeasurement(
        lot_id="NPI_LOT_01",  # 格式錯誤
        wafer_no=99,  # 超出 1~25 範圍
        die_x=0,
        die_y=0,
        frequency_ghz=-5.0,  # 頻率為負數
        leakage_current_ma=0.1,
    )
except Exception as e:
    print(f"成功攔截非法數據：\n{e}")

### 2. 多來源欄位標準化對齊 (standardize_columns)

- 實作：
    - 在測試整合工程中，不同廠牌機台（如 Advantest 與 Teradyne）或不同的外包封測廠（OSAT），匯出的 CSV 檔標頭（Headers）往往不統一。例如：有的人寫 WaferNum，有的人寫 WF_NO。
    - 我們必須編寫一個 standardize_columns() 轉換函數，搭配 Pydantic 將多來源欄位一鍵對齊。


In [ ]:
import pandas as pd

# --- 模擬來自兩個不同資料源的原始髒資料 ---
# 來源一：A 測試廠 (OSAT_A)
raw_df_A = pd.DataFrame(
    {
        "LotID": ["LOT_A01", "LOT_A01"],
        "WaferNum": [1, 1],
        "Freq_GHz": [28.1, 27.9],
        "Leakage": [0.02, 0.03],
    }
)

# 來源二：B 測試廠 (OSAT_B)
raw_df_B = pd.DataFrame(
    {"lot_no": ["LOT_B02"], "WF_NO": [2], "FREQUENCY": [24.5], "leak_ma": [0.12]}
)

# 定義統一對齊的欄位映射字典 (Mapping Dictionary)
COLUMN_MAPPING = {
    # 批號對齊
    "lotid": "lot_id",
    "lot_no": "lot_id",
    "lot_num": "lot_id",
    # 晶圓號對齊
    "wafernum": "wafer_no",
    "wf_no": "wafer_no",
    "wafer_id": "wafer_no",
    # 頻寬對齊
    "freq_ghz": "frequency_ghz",
    "frequency": "frequency_ghz",
    # 漏電流對齊
    "leakage": "leakage_current_ma",
    "leak_ma": "leakage_current_ma",
}


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """對應專案邏輯：將 DataFrame 的欄位標頭全轉小寫並精準映射對齊"""
    df_copied = df.copy()
    # 將現有欄位全部轉換為小寫，消除大小寫不一致的干擾
    df_copied.columns = [col.lower().strip() for col in df_copied.columns]
    # 進行對齊映射
    df_copied = df_copied.rename(columns=COLUMN_MAPPING)
    return df_copied


# --- 實作整併與 Pydantic 批次校驗 Pipeline ---
def data_ingestion_pipeline(raw_df: pd.DataFrame) -> List[DeviceTestMeasurement]:
    """清洗欄位並將每行轉為 Pydantic 驗證物件"""
    standard_df = standardize_columns(raw_df)
    valid_records = []

    # 將 DataFrame 的每一行轉為字典投餵給 Pydantic
    for _, row in standard_df.iterrows():
        # 由於 df.iterrows() 可能會改變部分原生型別，丟入 dict 前可做基礎轉換
        row_dict = row.to_dict()
        # 補上必要的座標預設值（若原始資料沒有）
        if "die_x" not in row_dict:
            row_dict["die_x"] = 0
        if "die_y" not in row_dict:
            row_dict["die_y"] = 0

        try:
            # 透過 Pydantic 強制進行 Runtime 校驗與型別對齊
            validated_model = DeviceTestMeasurement(**row_dict)
            valid_records.append(validated_model)
        except Exception as e:
            print(f"警告：資料列轉型校驗失敗，已自動剔除。錯誤原因: {e}")

    return valid_records


# --- 執行 Pipeline ---
print("=" * 50)
print(">>> 處理 A 廠數據：")
records_A = data_ingestion_pipeline(raw_df_A)
print(f"成功導入 {len(records_A)} 筆標準化資料。範例物件: {records_A[0]}")

print("\n>>> 處理 B 廠數據：")
records_B = data_ingestion_pipeline(raw_df_B)
print(f"成功導入 {len(records_B)} 筆標準化資料。範例物件: {records_B[0]}")
print("=" * 50)

- 總結：在柔性測試整合工程中，我們面對的是高度異質性的資料來源。不同的測試機台、不同的廠區所產生的原始 CSV 欄位名稱和大小寫常常各行其道。為了不讓這些髒資料污染後端的機器學習模型，我在資料流的最前線設計了數據守門員。首先，我編寫了 standardize_columns() 函數，利用統一的欄位映射字典，將所有異質欄位一鍵對齊。接著，我利用 Pydantic V2 來定義嚴格的 BaseModel。它不僅能幫我把字串自動轉成標準型別，更能透過 @field_validator 與 @model_validator 進行商業邏輯檢查（例如：限制晶圓號範圍在 1~25 之間，以及交叉檢查高漏電與高頻率的矛盾數據）。通過這套 Pipeline，不合格的髒資料會在最上游被徹底阻擋並拋出日誌警告，確保整體的穩健性。
